## 0. Summarize job ads
### a) Read all the jobs ads into python

In [1]:
import os

ads = {}

for filename in os.listdir('data'):
    if filename.endswith('.txt'):
        with open (f'data/{filename}', 'r', encoding='utf-8')as file:
            content = file.read()
            ads[filename] = content


### b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [2]:
from pydantic_ai import Agent

agent = Agent(
    "openrouter:nvidia/nemotron-3-nano-30b-a3b:free",
    system_prompt="""
Summarize the job ad into bullet points.

Format EXACTLY:

- Role: (specific job title, not company name, not generic like "Consultant")
- Responsibilities: (only clearly stated tasks)
- Required skills: (technical and relevant skills)
- Experience level: (extract exact years if mentioned, otherwise write "Not specified")

Rules:
- Do NOT guess or invent information
- Use the most specific role mentioned (e.g. "Data Scientist", not "Consultant")
- If experience is mentioned, ALWAYS include it
- Keep it concise and accurate
"""
)


# Define a function that takes ONE job ad as input
async def summarize_ad(job):
    summary = await agent.run(
        job
    )  # Call the AI and get a summary of the job ad  #gets result from AI
    return summary  # Return the result so we can use it outside the function

In [3]:
for name, ad in ads.items():  # Loop through all job ads in the dictionary
    # .items() means:“Give me both the key AND the value”
    # Call the function and send the job ad text (ad)
    # Store the returned result in 'summary'
    summary = await summarize_ad(ad)  #receives returned result
    print(name)
    print(summary.output)   # Print the actual summary text from the AI
    print()

job_ad1.txt
- Role: Data Engineer- Responsibilities: Take the lead on setting integration patterns from sources and building/migrating source integrations; be part of the team setting up and maintaining data platform infrastructure; involved in initiatives to introduce data product principles such as discoverability and trustworthiness; Design and build data models that support business processes ensuring governance and trust; Work with data transformation and modelling techniques using DBT with SQL and Python; Contribute to analytical projects to answer hypotheses or solve stakeholder ideas  
- Required skills: Building a modern data platform on GCP and Snowflake; Creating data platform design patterns and implementing them; Integrating data from sources such as APIs, MongoDB, MySQL; Setting up CI/CD pipelines and employing Infrastructure as Code; Orchestrating data pipelines with Airflow or similar; Modeling data with DBT and SQL; Programming with Python (or Java or Go); Experience w

### c) Now create and export markdown files for each job ad and its corresponding summary.



In [5]:
for name, ad in ads.items():
    summary = await summarize_ad(ad)
    md_filename = name.replace('.txt','.md')

    with open(f'data/{md_filename}', 'w', encoding='utf-8')as file:
        file.write(f'{name}\n\n## Summary\n\n{summary.output}')

### d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

I tested the function on another job ads from Arbetsförmedlingen. It was able to generate structured summaries with role, responsibilities, skills, and experience.

The function works best on clear and well-structured job ads. In some cases, the summaries are slightly too general or uneven in detail, but overall the results are accurate and useful.

## 1. Product extractor

### a) Read the products.json and store all of its descriptions into a list

In [1]:
import json

products =[]

with open('data/products.json', 'r') as f:
   products =json.load(f)

descriptions = [p['description'] for p in products]

In [2]:
descriptions

['The Nike Running Shoes X90 are available for $120.00 in the Footwear section.',
 'Grab the Sony WH-1000XM5 Headphones for just $349.99 â€” found in the Electronics department.',
 'The Dyson V12 Vacuum Cleaner is out of stock but available for pre-order at $599.99 in the Home & Garden section.',
 'Priced at $189.95, the Adidas Ultraboost 22 Sneakers are available in our Footwear collection.',
 'You can find the Apple iPad Pro 11-inch for $799.00 in the Electronics aisle.',
 "The Levi's 501 Original Jeans are currently available for $69.50 under our Clothing department.",
 'The Bosch Cordless Drill GSR18 is out of stock but available for pre-order at $149.00 in the Automotive section.',
 'Grab the Canon EOS R50 Camera for just $679.99 â€” available in the Electronics section.',
 'Priced at $299.95, the Philips Air Purifier 3000i is available in our Home & Garden collection.',
 'The Samsung Galaxy Watch 6 is out of stock but available for pre-order at $299.00 in the Electronics departme

### b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from typing import Literal

class Product(BaseModel):
    name: str
    price : float
    category : Literal[
        "Footwear",
        "Electronics",
        "Home & Garden",
        "Clothing",
        "Automotive"
    ]
    in_stock : bool
    description : str

agent = Agent('openrouter:nvidia/nemotron-3-super-120b-a12b:free', output_type= Product )

results = []

for desc in descriptions:
    result = await agent.run(desc)
    results.append(result)


In [ ]:
print(result.output.model_dump())

{'name': 'Samsung Galaxy Watch 6', 'price': 299.0, 'category': 'Electronics', 'in_stock': False, 'description': 'The Samsung Galaxy Watch 6 is out of stock but available for pre-order at $299.00 in the Electronics department.'}


In [23]:
results[2].output.model_dump()['name']

'Dyson V12 Vacuum Cleaner'